# isitfair hardening: `oracle_sparse`

Step-by-step audit of the `oracle_sparse` prediction table built by
`hardening/datasets/registry.py` (see `hardening/datasets/CARDS.md`). The
audited model is the logistic-regression column `y_proba_lr`; every stage is
its own cell, timed with `perf_counter`. Warnings and exceptions are recorded
per stage in `RECORD` and written to `hardening/notebooks/results/`.

Environment knobs: `HARDEN_B` (bootstrap replicates, default 50) and
`HARDEN_STRICT` (`1` makes the last cell fail on any unexpected error).

In [1]:
import faulthandler
import json
import os
import sys
import time
import traceback
import warnings
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import pandas as pd

for _c in (Path.cwd(), *Path.cwd().parents):
    if (_c / "hardening" / "datasets" / "registry.py").exists():
        REPO = _c
        break
sys.path.insert(0, str(REPO / "hardening" / "datasets"))
# Inside a Jupyter kernel sys.stderr has no file descriptor; give the watchdog a real file.
_FH = open(REPO / "hardening" / "logs" / "faulthandler_oracle_sparse.log", "w")
faulthandler.dump_traceback_later(600, repeat=True, file=_FH)
import registry  # noqa: E402

import isitfair  # noqa: E402
from isitfair import FairnessAudit, GroupRecalibration, WassersteinPostprocessing, report_to_pdf  # noqa: E402

NAME = "oracle_sparse"
B = int(os.environ.get("HARDEN_B", "50"))
STRICT = os.environ.get("HARDEN_STRICT", "0") == "1"
SEED = 42
OUT = REPO / "hardening" / "notebooks" / "results"
REPORTS = REPO / "hardening" / "reports"
OUT.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)
META = registry.meta(NAME)
THRESHOLD = META["threshold"]
RECORD = {"dataset": NAME, "B": B, "isitfair": isitfair.__version__,
          "isitfair_file": isitfair.__file__, "stages": {}, "variants": {}}
print(f"isitfair {isitfair.__version__} from {isitfair.__file__}")
print(f"{NAME}: B={B} threshold={THRESHOLD} strict={STRICT}")


@contextmanager
def stage(name, expect_error=False):
    """Time a stage; record its warnings and any exception, then carry on."""
    entry = {"seconds": None, "warnings": [], "error": None, "expected_error": expect_error}
    RECORD["stages"][name] = entry
    t0 = time.perf_counter()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        try:
            yield entry
        except Exception as exc:  # recorded, not raised, so later stages still run
            entry["error"] = f"{type(exc).__name__}: {exc}"
            entry["traceback"] = traceback.format_exc()
            print(f"!! {name} raised {entry['error']}")
    entry["seconds"] = round(time.perf_counter() - t0, 3)
    seen = {}
    for w in caught:
        key = f"{w.category.__name__}: {str(w.message)[:300]}"
        seen[key] = seen.get(key, 0) + 1
    entry["warnings"] = [f"{k} (x{v})" for k, v in seen.items()]
    print(f"[{name}] {entry['seconds']:.2f} s, {len(caught)} warning(s)"
          + (f", ERROR {entry['error']}" if entry["error"] else ""))
    for k in entry["warnings"][:8]:
        print("   ", k)
    (OUT / f"{NAME}_B{B}.json").write_text(json.dumps(RECORD, indent=2, default=str))

isitfair 0.4.0 from C:\Users\hugoa\Desktop\isitfair-golden\src\isitfair\__init__.py
oracle_sparse: B=50 threshold=0.15 strict=False


## 1. Load and validate inputs

In [2]:
with stage("load_validate"):
    df = registry.load(NAME)
    ATTRS = registry.ATTRS[NAME]
    checks = {
        "n": len(df),
        "index_is_range": isinstance(df.index, pd.RangeIndex),
        "y_true_dtype": str(df["y_true"].dtype),
        "y_true_values": sorted(df["y_true"].dropna().unique().tolist()),
        "y_true_missing": int(df["y_true"].isna().sum()),
        "y_proba_range": [float(df["y_proba_lr"].min()), float(df["y_proba_lr"].max())],
        "y_proba_missing": int(df["y_proba_lr"].isna().sum()),
        "attr_dtypes": {a: str(df[a].dtype) for a in ATTRS},
        "attr_missing": {a: int(df[a].isna().sum()) for a in ATTRS},
        "attr_levels": {a: int(df[a].nunique()) for a in ATTRS},
    }
    RECORD["checks"] = checks
    print(json.dumps(checks, indent=1))

{
 "n": 5000,
 "index_is_range": true,
 "y_true_dtype": "int64",
 "y_true_values": [
  0,
  1
 ],
 "y_true_missing": 0,
 "y_proba_range": [
  0.0028761774326127,
  0.7779995267471032
 ],
 "y_proba_missing": 0,
 "attr_dtypes": {
  "level": "object"
 },
 "attr_missing": {
  "level": 0
 },
 "attr_levels": {
  "level": 6
 }
}
[load_validate] 0.03 s, 0 warning(s)


A missing subgroup label must be rejected with an actionable message. If any
attribute has missing labels, try the raw column first (recorded), then
recode to an explicit `"Unknown"` category for the rest of the notebook.

In [3]:
missing_attrs = [a for a in ATTRS if df[a].isna().any()]
if missing_attrs:
    with stage("raw_missing_labels", expect_error=True):
        FairnessAudit(y_true=df["y_true"], y_proba=df["y_proba_lr"],
                      sensitive_features={a: df[a] for a in ATTRS},
                      threshold=THRESHOLD, n_bootstrap=B, random_state=SEED)
    for a in missing_attrs:
        df[a] = df[a].fillna("Unknown")
print("recoded to 'Unknown':", missing_attrs or "none")

recoded to 'Unknown': none


In [4]:
y = df["y_true"].to_numpy(dtype=float)
p = df["y_proba_lr"].to_numpy(dtype=float)
marginals = {a: df[a].to_numpy(dtype=object) for a in ATTRS}
INTERSECTION = tuple(ATTRS[:2]) if len(ATTRS) >= 2 else None
sf = dict(marginals)
if INTERSECTION:
    sf[INTERSECTION] = None
print("intersection:", INTERSECTION)

intersection: None


## 2. Construct the audit

In [5]:
with stage("construct"):
    audit = FairnessAudit(y_true=y, y_proba=p, sensitive_features=sf, threshold=THRESHOLD,
                          n_bootstrap=B, random_state=SEED)
    print(audit)

FairnessAudit(n=5000, events=593, threshold=0.15, n_bootstrap=50, random_state=42, attributes: level (6))
[construct] 0.01 s, 0 warning(s)


## 3. Discrimination

In [6]:
with stage("discrimination"):
    disc = audit.discrimination()
    print(disc[disc.subgroup != "GAP"][["attribute", "subgroup", "n", "n_events", "auroc",
                                         "auroc_ci_low", "auroc_ci_high"]].to_string(index=False))

attribute subgroup      n  n_events    auroc  auroc_ci_low  auroc_ci_high
    level  Overall 5000.0     593.0 0.747395      0.733204       0.761875
    level       L1 1800.0     211.0 0.755511      0.728806       0.782005
    level       L2 1200.0     176.0 0.736667      0.705777       0.768762
    level       L3  900.0     120.0 0.746741      0.703187       0.792709
    level       L4  600.0      83.0 0.763837      0.711206       0.817849
    level       L5  300.0       3.0 0.805836           NaN            NaN
    level       L6  200.0       0.0      NaN           NaN            NaN
[discrimination] 2.78 s, 0 warning(s)


## 4. Calibration

In [7]:
with stage("calibration"):
    cal = audit.calibration()
    print(cal[cal.subgroup != "GAP"][["attribute", "subgroup", "n", "calibration_intercept",
                                       "calibration_slope", "brier", "ece", "ici"]].to_string(index=False))

attribute subgroup      n  calibration_intercept  calibration_slope    brier      ece      ici
    level  Overall 5000.0              -0.131320           0.988379 0.094491 0.018980 0.013139
    level       L1 1800.0              -0.126935           1.018455 0.092906 0.016915 0.012447
    level       L2 1200.0               0.118418           0.961712 0.112165 0.029853 0.016940
    level       L3  900.0              -0.000118           0.981416 0.104520 0.027710 0.007331
    level       L4  600.0               0.061225           1.097783 0.101573 0.027144 0.017924
    level       L5  300.0              -2.857010           0.980583 0.035015 0.121890 0.122115
    level       L6  200.0                    NaN                NaN 0.025542 0.122318 0.122318
[calibration] 7.05 s, 0 warning(s)


## 5. Calibration curves

In [8]:
with stage("calibration_curves"):
    curves = audit.calibration_curves()
    print(curves.shape)

(700, 7)
[calibration_curves] 83.22 s, 0 warning(s)


## 6. ROC curves

In [9]:
with stage("roc_curves"):
    roc = audit.roc_curves()
    print(roc.shape)

(700, 7)
[roc_curves] 1.05 s, 0 warning(s)


## 7. Decision curve

In [10]:
with stage("decision_curve"):
    dc = audit.decision_curve()
    print(dc[dc.subgroup == "GAP"][["attribute", "gap_value", "gap_ci_low", "gap_ci_high"]].to_string(index=False))

attribute  gap_value  gap_ci_low  gap_ci_high
    level   0.149745         NaN          NaN
[decision_curve] 2.40 s, 0 warning(s)


## 8. Estimability

In [11]:
with stage("estimability"):
    est = audit.estimability()
    print(f"{int(est.not_estimable_final.sum())} of {len(est)} cell-metrics not reportable")

28 of 84 cell-metrics not reportable
[estimability] 0.04 s, 0 warning(s)


## 9. Hosmer-Lemeshow

In [12]:
with stage("hosmer_lemeshow"):
    hl = audit.hosmer_lemeshow()
    print(hl[hl.subgroup != "GAP"][["attribute", "subgroup", "n", "hl_statistic", "hl_df",
                                     "hl_p_value", "n_bins_used"]].to_string(index=False))

attribute subgroup      n  hl_statistic  hl_df  hl_p_value  n_bins_used
    level  Overall 5000.0     21.974715   10.0    0.015234         10.0
    level       L1 1800.0      7.112641   10.0    0.714770         10.0
    level       L2 1200.0     26.841512   10.0    0.002759         10.0
    level       L3  900.0     10.977110   10.0    0.359304         10.0
    level       L4  600.0      7.311738   10.0    0.695724         10.0
    level       L5  300.0     44.739830   10.0    0.000002         10.0
    level       L6  200.0           NaN    NaN         NaN         10.0
[hosmer_lemeshow] 2.75 s, 0 warning(s)


## 10. Cell diagnostics for one intersection

In [13]:
with stage("cell_diagnostics"):
    if INTERSECTION:
        cells = audit.cell_diagnostics()
        print(cells.to_string(index=False))
    else:
        print("single attribute; no intersection to diagnose")

single attribute; no intersection to diagnose
[cell_diagnostics] 0.00 s, 0 warning(s)


## 11. Recalibration ladder on the attribute with the largest intercept gap

In [14]:
with stage("recalibration_ladder"):
    gaps = cal[(cal.subgroup == "GAP") & (cal.gap_metric == "max_intercept_gap")
               & cal.attribute.isin(ATTRS)]
    LADDER_BY = str(gaps.sort_values("gap_value", ascending=False).attribute.iloc[0])
    ladder = audit.recalibration_ladder(by=LADDER_BY)
    print("by", LADDER_BY)
    print(ladder.to_string(index=False)[:3000])

!! recalibration_ladder raised ValueError: fold 0, stratum 'L6': zero events in the training union. Isotonic would silently return a constant and Platt would raise; neither may be papered over. Use fewer folds, or drop the stratum.
[recalibration_ladder] 0.03 s, 0 warning(s), ERROR ValueError: fold 0, stratum 'L6': zero events in the training union. Isotonic would silently return a constant and Platt would raise; neither may be papered over. Use fewer folds, or drop the stratum.


## 12. Mitigation on a held-out calibration split

support2 uses the committed calibration partition. Every other dataset splits
its audited test half 50/50 (stratified on the outcome, `random_state=42`):
the calibrators are fitted on one half and the before/after audit runs on the
other, so no row is scored by a calibrator fitted on it.

In [15]:
with stage("mitigation_split"):
    if NAME == "support2":
        cal_df = pd.read_csv(registry.PRED / "support2_calib.csv", dtype={a: object for a in ATTRS})
        eval_df = df
    else:
        from sklearn.model_selection import train_test_split
        i_cal, i_eval = train_test_split(np.arange(len(df)), test_size=0.5, random_state=SEED,
                                         stratify=df["y_true"])
        cal_df, eval_df = df.iloc[i_cal].reset_index(drop=True), df.iloc[i_eval].reset_index(drop=True)
    for a in ATTRS:
        cal_df[a] = cal_df[a].fillna("Unknown")
    calib_data = (cal_df["y_true"].to_numpy(float), cal_df["y_proba_lr"].to_numpy(float),
                  {a: cal_df[a].to_numpy(object) for a in ATTRS})
    sf_eval = {a: eval_df[a].to_numpy(object) for a in ATTRS}
    if INTERSECTION:
        sf_eval[INTERSECTION] = None
    audit_eval = audit if NAME == "support2" else FairnessAudit(
        y_true=eval_df["y_true"].to_numpy(float), y_proba=eval_df["y_proba_lr"].to_numpy(float),
        sensitive_features=sf_eval, threshold=THRESHOLD, n_bootstrap=B, random_state=SEED)
    print(f"calibration n={len(cal_df)}, evaluation n={len(eval_df)}")

calibration n=2500, evaluation n=2500
[mitigation_split] 0.02 s, 0 warning(s)


In [16]:
with stage("mitigate_group_recalibration"):
    mitigated_gr = audit_eval.mitigate(method=GroupRecalibration(method="isotonic"),
                                       calibration_data=calib_data)
    a0 = ATTRS[0]
    keep = ["subgroup", "calibration_intercept", "calibration_slope", "brier", "ece"]
    before, after = audit_eval.calibration(), mitigated_gr[a0].calibration()
    print("BEFORE", a0); print(before[before.attribute == a0][keep].to_string(index=False))
    print("AFTER (group isotonic)"); print(after[after.attribute == a0][keep].to_string(index=False))

BEFORE level
subgroup  calibration_intercept  calibration_slope    brier      ece
 Overall              -0.107637           0.999906 0.093848 0.013761
      L1              -0.128304           1.002170 0.092490 0.027750
      L2               0.086870           1.059958 0.105622 0.034061
      L3               0.079094           0.980373 0.108224 0.042468
      L4               0.230962           1.071928 0.109589 0.047028
      L5              -3.315911           0.684482 0.032757 0.122330
      L6                    NaN                NaN 0.023374 0.116471
     GAP                    NaN                NaN      NaN      NaN
     GAP                    NaN                NaN      NaN      NaN
AFTER (group isotonic)
subgroup  calibration_intercept  calibration_slope    brier      ece
 Overall               0.042562           0.370650 0.091816 0.017614
      L1              -0.025397           0.311639 0.092758 0.019863
      L2              -0.069135           0.538526 0.106286 0.01993

In [17]:
with stage("mitigate_wasserstein"):
    mitigated_w = audit_eval.mitigate(method=WassersteinPostprocessing(random_state=SEED),
                                      calibration_data=calib_data)
    d_after = mitigated_w[a0].discrimination()
    print(d_after[(d_after.attribute == a0) & (d_after.subgroup == "GAP")][
        ["gap_metric", "gap_value", "gap_ci_low", "gap_ci_high"]].to_string(index=False))

                  gap_metric  gap_value  gap_ci_low  gap_ci_high
equal_opportunity_difference   0.467742         NaN          NaN
         predictive_equality   0.069541         NaN          NaN
          equalized_odds_gap   0.467742         NaN          NaN
          statistical_parity   0.058357         NaN          NaN
[mitigate_wasserstein] 3.27 s, 0 warning(s)


## 13. Report: HTML, then PDF

In [18]:
REPORT = REPORTS / f"{NAME}_B{B}.html"
with stage("report_html"):
    audit_eval.report(out=REPORT, title=f"{NAME}: hardening audit (B={B})",
                      metadata={"Dataset": NAME, "Model": "LogisticRegression (registry)",
                                "Threshold": str(THRESHOLD), "Bootstrap": f"B={B}, random_state={SEED}"},
                      mitigation_audits=mitigated_gr)
    print(REPORT, REPORT.stat().st_size, "bytes")

C:\Users\hugoa\Desktop\isitfair\hardening\reports\oracle_sparse_B50.html 621492 bytes
[report_html] 46.85 s, 0 warning(s)


In [19]:
with stage("report_pdf"):
    pdf_path = report_to_pdf(REPORT)
    print(pdf_path, pdf_path.stat().st_size, "bytes")

C:\Users\hugoa\Desktop\isitfair\hardening\reports\oracle_sparse_B50.pdf 211790 bytes
[report_pdf] 17.58 s, 0 warning(s)


## 14. Input-type variants

The same audit (B = 20 to keep this cheap) with inputs passed in the forms
real pipelines produce. Each variant is classified as `raises`, `warns`,
`ok-identical` (results equal the plain-numpy baseline exactly), or
`ok-DIFFERENT` (runs silently but gives different numbers: a silent
misbehaviour unless the input itself changed, as in the ties variant).

In [20]:
VB = 20


def run_variant(name, y_in, p_in, sf_in, compare=True, ref=None):
    out = {"status": None, "warnings": [], "error": None}
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        try:
            a = FairnessAudit(y_true=y_in, y_proba=p_in, sensitive_features=sf_in,
                              threshold=THRESHOLD, n_bootstrap=VB, random_state=SEED)
            frames = {"discrimination": a.discrimination(), "calibration": a.calibration(),
                      "hosmer_lemeshow": a.hosmer_lemeshow(), "decision_curve": a.decision_curve()}
        except Exception as exc:
            out["error"] = f"{type(exc).__name__}: {exc}"
            frames = None
    out["warnings"] = sorted({f"{w.category.__name__}: {str(w.message)[:200]}" for w in caught})
    if frames is None:
        out["status"] = "raises"
    else:
        num_bad = {k: int(np.isinf(f.select_dtypes("number").to_numpy(float)).sum()) for k, f in frames.items()}
        out["inf_values"] = num_bad
        if ref is not None and compare:
            same = all(frames[k].equals(ref[k]) for k in frames)
            out["identical_to_baseline"] = same
            out["status"] = ("warns, " if out["warnings"] else "") + ("ok-identical" if same else "ok-DIFFERENT")
            if not same:
                diffs = {}
                for k in frames:
                    fa, fb = frames[k], ref[k]
                    if fa.shape == fb.shape:
                        na = fa.select_dtypes("number").to_numpy(float)
                        nb = fb.select_dtypes("number").to_numpy(float)
                        diffs[k] = float(np.nanmax(np.abs(na - nb))) if na.size else 0.0
                    else:
                        diffs[k] = f"shape {fa.shape} vs {fb.shape}"
                out["max_abs_diff"] = diffs
        else:
            out["status"] = ("warns, " if out["warnings"] else "") + "ok"
    RECORD["variants"][name] = out
    print(f"{name:<28} {out['status']}" + (f"  {out['error']}" if out["error"] else ""))
    for w in out["warnings"][:5]:
        print("     ", w)
    (OUT / f"{NAME}_B{B}.json").write_text(json.dumps(RECORD, indent=2, default=str))
    return frames


with stage("variants"):
    base = run_variant("baseline_numpy", y, p, sf, compare=False)
    s_attr = {a: pd.Series(v, dtype="category") for a, v in marginals.items()}
    run_variant("category_attributes", y, p, {**s_attr, **({INTERSECTION: None} if INTERSECTION else {})}, ref=base)
    run_variant("Int64_outcome", pd.Series(y.astype(int), dtype="Int64"), p, sf, ref=base)
    run_variant("bool_outcome", y.astype(bool), p, sf, ref=base)
    run_variant("bool_outcome_series", pd.Series(y.astype(bool)), p, sf, ref=base)
    perm = np.random.default_rng(SEED).permutation(len(y))
    idx = pd.Index(perm + 10_000)
    run_variant("shuffled_index_consistent",
                pd.Series(y, index=idx), pd.Series(p, index=idx),
                {**{a: pd.Series(v, index=idx) for a, v in marginals.items()},
                 **({INTERSECTION: None} if INTERSECTION else {})}, ref=base)
    # y sorted/shuffled with its labels, p left in the original order: the
    # classic silent misalignment, which isitfair is documented to reject.
    run_variant("shuffled_index_misaligned",
                pd.Series(y).iloc[perm], pd.Series(p), sf, ref=base)
    p_ties = np.round(p, 2)
    p_ties[:: max(1, len(p) // 50)] = 0.0
    p_ties[1:: max(1, len(p) // 50)] = 1.0
    run_variant("ties_and_exact_0_1", y, p_ties, sf, compare=False)

baseline_numpy               ok


category_attributes          ok-identical


Int64_outcome                ok-identical


bool_outcome                 ok-identical


bool_outcome_series          ok-identical


shuffled_index_consistent    ok-identical
shuffled_index_misaligned    raises  ValueError: `y_proba` and `y_true` are pandas objects with the same index labels in a different order. FairnessAudit aligns inputs by position, not by index, so the rows would be mismatched. Reorder them consistently, e.g. `s.loc[other.index]`, or pass `.to_numpy()` arrays built from one DataFrame.


ties_and_exact_0_1           ok
[variants] 77.19 s, 0 warning(s)


## 15. Summary

In [21]:
unexpected = {k: v["error"] for k, v in RECORD["stages"].items()
              if v["error"] and not v["expected_error"]}
RECORD["unexpected_errors"] = unexpected
RECORD["total_seconds"] = round(sum(v["seconds"] or 0 for v in RECORD["stages"].values()), 2)
(OUT / f"{NAME}_B{B}.json").write_text(json.dumps(RECORD, indent=2, default=str))
print(pd.DataFrame({k: {"seconds": v["seconds"], "n_warnings": len(v["warnings"]),
                        "error": v["error"]} for k, v in RECORD["stages"].items()}).T.to_string())
print("unexpected errors:", unexpected or "none")
if STRICT:
    assert not unexpected, unexpected

                             seconds n_warnings                                                                                                                                                                                                     error
load_validate                  0.028        0.0                                                                                                                                                                                                       NaN
construct                      0.015        0.0                                                                                                                                                                                                       NaN
discrimination                 2.782        0.0                                                                                                                                                                                                       NaN
